# 16 - Interpretability and Explainability

In this notebook, we crack open the Black Box. We will use Permutation Importance and SHAP to explain exactly *why* a model made a specific prediction.

## Concept Overview
* **Global Explainability**: How does the model work overall? Which features are most important?
* **Local Explainability**: Why did the model make *this specific decision* for *this specific row*?

## 1. Global Explainability: Permutation Importance
Permutation Importance works by shuffling a column and seeing how much the accuracy drops. If accuracy drops massively, the feature was highly important.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.inspection import permutation_importance

# Load Data
data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = data.target

X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)

# Train Model
model = RandomForestClassifier(random_state=42)
model.fit(X_train, y_train)

# Calculate Permutation Importance on the TEST set
result = permutation_importance(model, X_test, y_test, n_repeats=10, random_state=42)

# Sort and Plot Top 10
sorted_idx = result.importances_mean.argsort()[-10:]
plt.figure(figsize=(10, 6))
plt.boxplot(result.importances[sorted_idx].T, vert=False, labels=X.columns[sorted_idx])
plt.title("Permutation Importance (Test Set)")
plt.xlabel("Decrease in Accuracy Score")
plt.show()

## 2. Local Explainability: SHAP
SHAP (SHapley Additive exPlanations) is the industry standard. It tells you exactly how many 'points' each feature contributed to a specific prediction.

*Note: Ensure `shap` is installed (`pip install shap`)*.

In [ ]:
import shap
# Initialize JS for SHAP plots
shap.initjs()

# Create a SHAP explainer for the Random Forest
explainer = shap.TreeExplainer(model)

# Calculate SHAP values for the test set
shap_values = explainer.shap_values(X_test)

print("SHAP values calculated successfully.")

### Global SHAP Summary Plot
This plot shows the impact of every feature for every single patient in the dataset.

In [ ]:
# Plot summary (Class 1 - Benign)
# We use shap_values[1] because this is a binary classification and we want the SHAP values for the positive class.
shap.summary_plot(shap_values[1], X_test, plot_type="dot")

### Local SHAP Force Plot
Let's explain Patient #0. Why did the model predict what it predicted for this specific person?

In [ ]:
# Explain the very first patient in the test set
# Red arrows push the probability higher. Blue arrows push it lower.
shap.force_plot(explainer.expected_value[1], shap_values[1][0,:], X_test.iloc[0,:])

## Industry Notes
If your model rejects a user's mortgage application, regulators require you to provide specific reasons. SHAP local explainers are the exact mathematical tool used by financial institutions to generate these 'Adverse Action' reason codes.

## Exercises
1. Try installing `lime` and using `LimeTabularExplainer` on the same patient above. Does it agree with SHAP?
2. Look up `shap.dependence_plot`. Plot the dependence of the `worst area` feature.